# 🚘 AI Number Plate Recognition (ANPR)

Portfolio-ready computer vision project for **Automatic Number Plate Recognition**.

**Pipeline:** Vehicle Image → Plate Detection → Crop → Preprocessing → OCR → Text Cleaning → Validation → Result

The runnable demo uses a synthetic vehicle image. A production version can replace the baseline with YOLO and a real OCR engine.


## 🎯 Objectives

- Detect a plate-like region.
- Crop and preprocess the plate.
- Prepare the crop for OCR.
- Normalize recognized text.
- Validate a simplified Indian registration pattern.
- Visualize intermediate outputs.
- Define a YOLO + OCR production upgrade path.
- Include privacy and responsible-use considerations.


In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
import cv2

print("OpenCV version:", cv2.__version__)


## 1. 🖼️ Synthetic Vehicle Image


In [ ]:
canvas = np.full((420, 760, 3), 210, dtype=np.uint8)

cv2.rectangle(canvas, (90, 80), (670, 350), (70, 90, 120), -1)
cv2.rectangle(canvas, (170, 115), (590, 230), (120, 150, 170), -1)

plate_xyxy = (245, 270, 515, 330)
x1, y1, x2, y2 = plate_xyxy
cv2.rectangle(canvas, (x1, y1), (x2, y2), (245, 245, 245), -1)
cv2.rectangle(canvas, (x1, y1), (x2, y2), (20, 20, 20), 3)
cv2.putText(canvas, "TN 23 AB 1234", (260, 311),
            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (15, 15, 15), 2, cv2.LINE_AA)

plt.figure(figsize=(12, 6))
plt.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
plt.title("Synthetic Vehicle Image")
plt.axis("off")
plt.show()


## 2. 🔎 Plate Detection Baseline

The baseline uses grayscale conversion, denoising, edge detection, contours, aspect ratio and area filtering. Real-world systems should use a trained detector such as YOLO.


In [ ]:
gray = cv2.cvtColor(canvas, cv2.COLOR_BGR2GRAY)
blurred = cv2.bilateralFilter(gray, 11, 17, 17)
edges = cv2.Canny(blurred, 40, 160)

contours, _ = cv2.findContours(edges, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
candidates = []

for contour in contours:
    x, y, w, h = cv2.boundingRect(contour)
    area = w * h
    aspect = w / max(h, 1)
    if 2.5 <= aspect <= 6.5 and 5000 <= area <= 30000:
        candidates.append((area, x, y, w, h))

candidates = sorted(candidates, reverse=True)
print("Candidate regions:", len(candidates))

plt.figure(figsize=(12, 5))
plt.imshow(edges, cmap="gray")
plt.title("Edge Map")
plt.axis("off")
plt.show()


## 3. ✂️ Plate Crop


In [ ]:
if candidates:
    _, x, y, w, h = candidates[0]
    plate_crop = canvas[y:y+h, x:x+w]
    detected_box = (x, y, x+w, y+h)
else:
    x1, y1, x2, y2 = plate_xyxy
    plate_crop = canvas[y1:y2, x1:x2]
    detected_box = plate_xyxy

plt.figure(figsize=(8, 3))
plt.imshow(cv2.cvtColor(plate_crop, cv2.COLOR_BGR2RGB))
plt.title("Cropped Number Plate")
plt.axis("off")
plt.show()


## 4. 🧹 OCR Preprocessing


In [ ]:
plate_gray = cv2.cvtColor(plate_crop, cv2.COLOR_BGR2GRAY)
plate_gray = cv2.resize(plate_gray, None, fx=2.5, fy=2.5, interpolation=cv2.INTER_CUBIC)
plate_gray = cv2.GaussianBlur(plate_gray, (3, 3), 0)
_, plate_binary = cv2.threshold(
    plate_gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

plt.figure(figsize=(10, 3))
plt.imshow(plate_binary, cmap="gray")
plt.title("OCR-Ready Plate")
plt.axis("off")
plt.show()


## 5. 🔤 OCR + Text Normalization

For the reproducible demo, the expected synthetic plate text is used as mock OCR output. Connect Tesseract, EasyOCR or PaddleOCR to `plate_binary` for real images.


In [ ]:
raw_ocr_text = "TN 23 AB 1234"

def normalize_plate_text(text):
    return re.sub(r"[^A-Z0-9]", "", text.upper())

normalized_plate = normalize_plate_text(raw_ocr_text)
print("Raw OCR:", raw_ocr_text)
print("Normalized:", normalized_plate)


## 6. ✅ Simplified Indian Plate Validation

Real registration formats have legitimate variations, so production validation should support the formats relevant to the deployment jurisdiction.


In [ ]:
STANDARD_INDIA_PATTERN = re.compile(r"^[A-Z]{2}[0-9]{1,2}[A-Z]{1,3}[0-9]{1,4}$")

def validate_standard_plate(text):
    cleaned = normalize_plate_text(text)
    return {
        "plate": cleaned,
        "matches_simplified_standard_pattern":
            bool(STANDARD_INDIA_PATTERN.fullmatch(cleaned))
    }

validation = validate_standard_plate(raw_ocr_text)
validation


## 7. 📊 Structured Result


In [ ]:
result = {
    "plate_text": normalized_plate,
    "valid_pattern": validation["matches_simplified_standard_pattern"],
    "detector": "classical-cv-demo",
    "ocr": "synthetic-demo",
    "bounding_box": detected_box
}

result


## 8. 🤖 Production Architecture

```text
Camera / Image
      ↓
Quality Checks
      ↓
YOLO Plate Detector
      ↓
Plate Crop
      ↓
Perspective + Contrast Correction
      ↓
OCR Engine
      ↓
Text Normalization
      ↓
Format Validation
      ↓
Confidence Rules
      ↓
Structured Result / Human Review
```

Recommended upgrade: train YOLO on legally sourced annotated plate images and evaluate detection and OCR separately before measuring end-to-end recognition accuracy.


## 9. 🧪 Evaluation

### Detection
- Precision
- Recall
- mAP@50
- mAP@50:95
- IoU

### OCR
- Character Error Rate (CER)
- Character accuracy
- Exact plate match

### End-to-End
- Full-plate recognition rate
- Latency
- Review/failure rate
- Performance under blur, night, angle, distance and occlusion


## 10. 🛡️ Privacy & Responsible Use

- Use ANPR only for legitimate, authorized purposes.
- Minimize retention of raw images and plate records.
- Apply access controls and encryption to stored data.
- Keep audit logs for searches and exports.
- Avoid inferring vehicle-owner identity without lawful authorization.
- Use human review for uncertain predictions.
- Follow applicable privacy, surveillance, traffic and data-protection requirements.


## 11. 🚀 Roadmap

### Phase 1
- [x] Image preprocessing
- [x] Plate-region baseline
- [x] Cropping
- [x] OCR preprocessing
- [x] Text normalization
- [x] Simplified validation

### Phase 2
- [ ] YOLO plate detector
- [ ] Annotated dataset
- [ ] Data augmentation
- [ ] Detection evaluation

### Phase 3
- [ ] EasyOCR / PaddleOCR / Tesseract
- [ ] Perspective correction
- [ ] OCR confidence
- [ ] Character-error evaluation

### Phase 4
- [ ] FastAPI inference service
- [ ] React dashboard
- [ ] Image upload
- [ ] Authorized video processing
- [ ] Human review queue

### Phase 5
- [ ] Experiment tracking
- [ ] Dataset versioning
- [ ] Model monitoring
- [ ] CI/CD
- [ ] Secure deployment

## ✅ Conclusion

This notebook demonstrates the core ANPR workflow from **image → detection → crop → OCR preprocessing → text normalization → validation**. The portfolio-grade next step is YOLO + OCR evaluated on diverse, legally sourced real-world data.
